# Tarea 2 · Parte 2 — De los modelos base a la extracción estructurada

1. Muestra de evaluación: **10 documentos y 10 clips del corpus de la Parte 1**, con transcripción **manual**.
2. Línea base: PaddleOCR sin personalización (CER) y faster-whisper `small` en CPU (WER).
3. Extracción estructurada con un LLM local (Ollama) + auditoría humana de 20 casos.
4. Modelos alternativos sobre **las mismas** muestras: Tesseract y PaddleOCR con enderezado; faster-whisper turbo y Qwen3-ASR.
5. Casos difíciles, tabla comparativa y decisión.

**Regla de oro:** las configuraciones de todos los modelos se fijan y guardan en
`modelos/config_evaluacion.json` **antes** de calcular cualquier métrica, y no se tocan después.

## 1 · Muestra de evaluación
10 documentos (3 impresos, 3 fotos, 4 manuscritos) y 10 clips (3 entrevistas de Wikimedia Commons con ruido de calle + 7 clips de lectura de OpenSLR 73), elegidos con semilla fija. Se excluyeron documentos de más de 1 500 caracteres porque transcribirlos a mano no es viable; el criterio es la longitud, no el error de ningún modelo. La muestra y las configuraciones quedan fijadas en `modelos/config_evaluacion.json` antes de calcular cualquier métrica. Las transcripciones manuales se escriben en `datos/raw/referencias/` (no se publican).

In [1]:
import json
from datetime import date
from pathlib import Path
import pandas as pd

documentos = pd.read_csv("datos/documentos.csv")
clips = pd.read_csv("datos/transcripciones.csv")
SEMILLA = 2026

# Documentos: 3 impresos, 3 fotos, 4 manuscritos. Se excluyen los de más de 1500 caracteres
# (transcribir a mano un periódico entero no es viable). El criterio es el largo, NO el error del OCR.
documentos["n_car"] = documentos.texto_ocr.fillna("").str.len()
candidatos = documentos[documentos.n_car <= 1500]
muestra_docs = pd.concat([candidatos[candidatos.tipo == t].sample(k, random_state=SEMILLA)
                          for t, k in {"impreso": 3, "foto": 3, "manuscrito": 4}.items()]).id.tolist()

# Audio: 3 entrevistas de Commons (condiciones reales, con ruido) + 7 clips de lectura de OpenSLR
commons = clips[clips.fuente == "Wikimedia Commons"].sort_values("duracion_seg").head(3)
openslr = clips[(clips.fuente == "OpenSLR 73") & (clips.duracion_seg < 100)].sample(7, random_state=SEMILLA)
muestra_audio = pd.concat([commons, openslr]).id.tolist()

print("Documentos:", muestra_docs)
print("Audios:    ", muestra_audio)
print(f"Minutos de audio a transcribir: {clips.set_index('id').loc[muestra_audio, 'duracion_seg'].sum() / 60:.1f}")

# Se fija ANTES de ver cualquier métrica
Path("modelos").mkdir(exist_ok=True)
cfg = Path("modelos/config_evaluacion.json")
if not cfg.exists():
    cfg.write_text(json.dumps({"fecha": date.today().isoformat(), "semilla": SEMILLA,
                               "muestra_docs": muestra_docs, "muestra_audio": muestra_audio},
                              ensure_ascii=False, indent=1), encoding="utf-8")

# Plantillas para tu transcripción manual (en raw/: no se publican)
REF = Path("datos/raw/referencias"); REF.mkdir(parents=True, exist_ok=True)
documentos.set_index("id").loc[muestra_docs, ["tipo", "archivo"]].assign(texto_referencia="").to_csv(REF / "ref_documentos.csv")
clips.set_index("id").loc[muestra_audio, ["fuente", "archivo_audio", "duracion_seg"]].assign(texto_referencia="").to_csv(REF / "ref_audio.csv")
print("Plantillas creadas en", REF)

# Para adaptar las siguientes celdas a tus archivos:
for f in sorted(Path("datos/raw").glob("*.csv")):
    print(f.name, list(pd.read_csv(f, nrows=1).columns))

Documentos: ['doc_008', 'doc_002', 'doc_007', 'doc_014', 'doc_015', 'doc_016', 'doc_024', 'doc_020', 'doc_025', 'doc_022']
Audios:     ['clip_007', 'clip_002', 'clip_004', 'clip_021', 'clip_016', 'clip_020', 'clip_024', 'clip_009', 'clip_022', 'clip_018']
Minutos de audio a transcribir: 10.9
Plantillas creadas en datos\raw\referencias
fuentes_documentos.csv ['id;tipo;descripcion;fuente_url;licencia;autor;acceso;publicar_imagen']
ocr_crudo.csv ['id', 'tipo', 'descripcion', 'fuente_url', 'licencia', 'autor', 'acceso', 'publicar_imagen', 'sha256', 'texto_ocr_crudo', 'n_lineas', 'confianza_media', 'segundos_ocr', 'modelo_ocr', 'archivo']
reemplazos_privados.csv ['   nombre', 'seudonimo']


### Transcripción manual de referencia

Se crea un archivo `.txt` por cada documento y clip de la muestra en `datos/raw/referencias/docs/` y `datos/raw/referencias/audio/`, donde escribo a mano el texto de referencia. Uso `.txt` en lugar de editar el CSV directamente para conservar los saltos de línea y las tildes.

**Criterios de transcripción** (los mismos para todos los archivos):
- **Documentos:** copio el texto tal como aparece en la imagen, línea por línea, con la ortografía original y sin corregir errores. Si una palabra es ilegible, escribo mi mejor lectura.
- **Audios:** escribo todas las palabras habladas, sin marcas de hablante, risas ni ruidos. Los nombres de personas se escriben tal cual y luego se seudonimizan igual en la referencia y en la salida de los modelos.
- Transcribo **sin mirar la salida de ningún modelo**, para que la referencia sea independiente.

Después, estos textos se copian a la columna `texto_referencia` de `documentos.csv` y `transcripciones.csv`.

In [ ]:
#Un archivo de texto por cada referencia (más cómodo que escribir dentro de un CSV)
REF = Path("datos/raw/referencias")
for carpeta, ids in [("docs", muestra_docs), ("audio", muestra_audio)]:
    (REF / carpeta).mkdir(parents=True, exist_ok=True)
    for i in ids:
        f = REF / carpeta / f"{i}.txt"
        if not f.exists():
            f.write_text("", encoding="utf-8")
print("Archivos para transcribir en", REF / "docs", "y", REF / "audio")

Archivos para transcribir en datos\raw\referencias\docs y datos\raw\referencias\audio
